# Trapezium Signed-Distance Map

Minimal notebook.

Public functions:

```python
trapetium_border(s, TOP, BOTTOM, HEIGHT)
trapetium_map(s, d, TOP, BOTTOM, HEIGHT)
trapetium_map_inverse(x, y, TOP, BOTTOM, HEIGHT)
trapetium_jacobian(s, d, TOP, BOTTOM, HEIGHT)
trapetium_jacobian_inverse(s, d, TOP, BOTTOM, HEIGHT)
```

Here `s` is the modular perimeter fraction in `[0,1)`. The signed distance is `d > 0` inside, `d = 0` on the wall, and `d < 0` outside.


In [ ]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt

try:
    from numba import njit
except Exception:
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        def decorator(func):
            return func
        return decorator

FIGURE_DIR = Path("Figures")
FIGURE_DIR.mkdir(exist_ok=True)


In [ ]:
TOP = 2.3
BOTTOM = 4.0
HEIGHT = 2.7


In [ ]:
@njit(fastmath=True)
def _clamp(v, lo, hi):
    if v < lo:
        return lo
    if v > hi:
        return hi
    return v


@njit(fastmath=True)
def _dot2(x, y):
    return x * x + y * y


@njit(fastmath=True)
def _perimeter(TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    side = math.sqrt((r1 - r2) * (r1 - r2) + HEIGHT * HEIGHT)
    return BOTTOM + TOP + 2.0 * side


@njit(fastmath=True)
def _side_length(TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    return math.sqrt((r1 - r2) * (r1 - r2) + HEIGHT * HEIGHT)


@njit(fastmath=True)
def _edge_data(edge_id, TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    he = 0.5 * HEIGHT
    side = _side_length(TOP, BOTTOM, HEIGHT)

    if edge_id == 0:
        vx = -r1
        vy = -he
        tx = 1.0
        ty = 0.0
        length = BOTTOM
    elif edge_id == 1:
        vx = r1
        vy = -he
        tx = (r2 - r1) / side
        ty = HEIGHT / side
        length = side
    elif edge_id == 2:
        vx = r2
        vy = he
        tx = -1.0
        ty = 0.0
        length = TOP
    else:
        vx = -r2
        vy = he
        tx = (-r1 + r2) / side
        ty = -HEIGHT / side
        length = side

    # Inward normal for the counterclockwise boundary.
    nx = -ty
    ny = tx
    return vx, vy, tx, ty, nx, ny, length


@njit(fastmath=True)
def _edge_from_s(s, TOP, BOTTOM, HEIGHT):
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    side = _side_length(TOP, BOTTOM, HEIGHT)

    s_mod = s - math.floor(s)
    S = s_mod * perimeter

    if S < BOTTOM:
        return 0, S
    S -= BOTTOM
    if S < side:
        return 1, S
    S -= side
    if S < TOP:
        return 2, S
    S -= TOP
    return 3, S


@njit(fastmath=True)
def _interior_angle(vertex_id, TOP, BOTTOM, HEIGHT):
    prev_edge = vertex_id - 1
    if prev_edge < 0:
        prev_edge = 3
    next_edge = vertex_id

    _, _, tx_prev, ty_prev, _, _, _ = _edge_data(prev_edge, TOP, BOTTOM, HEIGHT)
    _, _, tx_next, ty_next, _, _, _ = _edge_data(next_edge, TOP, BOTTOM, HEIGHT)

    # Interior angle between the incoming and outgoing side directions.
    ux = -tx_prev
    uy = -ty_prev
    vx = tx_next
    vy = ty_next
    c = _clamp(ux * vx + uy * vy, -1.0, 1.0)
    return math.acos(c)


@njit(fastmath=True)
def _edge_s_interval(edge_id, d, TOP, BOTTOM, HEIGHT):
    """Valid nearest-side interval for plotting the side branch at distance d."""
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    side = _side_length(TOP, BOTTOM, HEIGHT)

    if edge_id == 0:
        S0 = 0.0
        length = BOTTOM
    elif edge_id == 1:
        S0 = BOTTOM
        length = side
    elif edge_id == 2:
        S0 = BOTTOM + side
        length = TOP
    else:
        S0 = BOTTOM + side + TOP
        length = side

    q = abs(d)
    start_vertex = edge_id
    end_vertex = edge_id + 1
    if end_vertex == 4:
        end_vertex = 0

    a0 = _interior_angle(start_vertex, TOP, BOTTOM, HEIGHT)
    a1 = _interior_angle(end_vertex, TOP, BOTTOM, HEIGHT)
    m0 = 0.0
    m1 = 0.0
    if q > 0.0:
        m0 = q / math.tan(0.5 * a0)
        m1 = q / math.tan(0.5 * a1)

    if m0 + m1 >= length:
        mid = S0 + 0.5 * length
        return mid / perimeter, mid / perimeter

    return (S0 + m0) / perimeter, (S0 + length - m1) / perimeter


@njit(fastmath=True)
def trapetium_border(s, TOP, BOTTOM, HEIGHT):
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    vx, vy, tx, ty, _, _, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    return vx + sigma * tx, vy + sigma * ty


@njit(fastmath=True)
def trapetium_map(s, d, TOP, BOTTOM, HEIGHT):
    """Side-branch forward map (s,d) -> (x,y), with positive d inside."""
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    vx, vy, tx, ty, nx, ny, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    px = vx + sigma * tx
    py = vy + sigma * ty
    return px + d * nx, py + d * ny


@njit(fastmath=True)
def _sd_trapetium_iq_positive_inside(x, y, TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    he = 0.5 * HEIGHT

    px = abs(x)
    py = y

    k1x = r2
    k1y = he
    k2x = r2 - r1
    k2y = 2.0 * he

    limit = r1
    if py >= 0.0:
        limit = r2

    ca_x = px - min(px, limit)
    ca_y = abs(py) - he

    h = ((k1x - px) * k2x + (k1y - py) * k2y) / _dot2(k2x, k2y)
    h = _clamp(h, 0.0, 1.0)
    cb_x = px - k1x + k2x * h
    cb_y = py - k1y + k2y * h

    sign = 1.0
    if cb_x < 0.0 and ca_y < 0.0:
        sign = -1.0

    d2a = _dot2(ca_x, ca_y)
    d2b = _dot2(cb_x, cb_y)
    if d2a < d2b:
        return -sign * math.sqrt(d2a)
    return -sign * math.sqrt(d2b)


@njit(fastmath=True)
def _project_segment(px, py, ax, ay, bx, by):
    ex = bx - ax
    ey = by - ay
    u = ((px - ax) * ex + (py - ay) * ey) / _dot2(ex, ey)
    u = _clamp(u, 0.0, 1.0)
    cx = ax + u * ex
    cy = ay + u * ey
    dx = px - cx
    dy = py - cy
    return _dot2(dx, dy), u


@njit(fastmath=True)
def trapetium_map_inverse(x, y, TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    he = 0.5 * HEIGHT
    side = _side_length(TOP, BOTTOM, HEIGHT)
    perimeter = BOTTOM + TOP + 2.0 * side

    ax = -r1
    ay = -he
    bx = r1
    by = -he
    cx = r2
    cy = he
    dx = -r2
    dy = he

    best_d2 = 1.0e300
    best_edge = 0
    best_u = 0.0

    d2, u = _project_segment(x, y, ax, ay, bx, by)
    if d2 < best_d2:
        best_d2 = d2
        best_edge = 0
        best_u = u

    d2, u = _project_segment(x, y, bx, by, cx, cy)
    if d2 < best_d2:
        best_d2 = d2
        best_edge = 1
        best_u = u

    d2, u = _project_segment(x, y, cx, cy, dx, dy)
    if d2 < best_d2:
        best_d2 = d2
        best_edge = 2
        best_u = u

    d2, u = _project_segment(x, y, dx, dy, ax, ay)
    if d2 < best_d2:
        best_d2 = d2
        best_edge = 3
        best_u = u

    if best_edge == 0:
        S = best_u * BOTTOM
    elif best_edge == 1:
        S = BOTTOM + best_u * side
    elif best_edge == 2:
        S = BOTTOM + side + best_u * TOP
    else:
        S = BOTTOM + side + TOP + best_u * side

    s = S / perimeter
    if s >= 1.0:
        s -= 1.0

    d = _sd_trapetium_iq_positive_inside(x, y, TOP, BOTTOM, HEIGHT)
    return s, d


@njit(fastmath=True)
def trapetium_jacobian(s, d, TOP, BOTTOM, HEIGHT):
    edge_id, _ = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    _, _, tx, ty, nx, ny, _ = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    return perimeter * tx, nx, perimeter * ty, ny


@njit(fastmath=True)
def trapetium_jacobian_det(s, d, TOP, BOTTOM, HEIGHT):
    j00, j01, j10, j11 = trapetium_jacobian(s, d, TOP, BOTTOM, HEIGHT)
    return j00 * j11 - j01 * j10


@njit(fastmath=True)
def trapetium_jacobian_inverse(s, d, TOP, BOTTOM, HEIGHT):
    j00, j01, j10, j11 = trapetium_jacobian(s, d, TOP, BOTTOM, HEIGHT)
    det = j00 * j11 - j01 * j10
    return j11 / det, -j01 / det, -j10 / det, j00 / det


@njit(fastmath=True)
def trapetium_jacobian_inverse_det(s, d, TOP, BOTTOM, HEIGHT):
    inv00, inv01, inv10, inv11 = trapetium_jacobian_inverse(s, d, TOP, BOTTOM, HEIGHT)
    return inv00 * inv11 - inv01 * inv10


## Plot The Boundary


In [ ]:
s_plot = np.linspace(0.0, 1.0, 1000)
P = np.array([trapetium_border(s, TOP, BOTTOM, HEIGHT) for s in s_plot])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(P[:, 0], P[:, 1], color="black", lw=1.5)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("trapetium_border(s)")
ax.grid(True)
fig.savefig(FIGURE_DIR / "trapetium_border.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot The Forward Map


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

d_values = [0.0, 0.15, 0.30, 0.45, 0.60, 0.75, 1.00, 2.00]

for d in d_values:
    for edge_id in range(4):
        s0, s1 = _edge_s_interval(edge_id, d, TOP, BOTTOM, HEIGHT)
        if s1 <= s0:
            continue
        ss = np.linspace(s0, s1, 250)
        xy = np.array([trapetium_map(s, d, TOP, BOTTOM, HEIGHT) for s in ss])
        label = f"d = {d:.2f}" if edge_id == 0 else None
        ax.plot(xy[:, 0], xy[:, 1], lw=1.0, label=label)

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("valid side branches of trapetium_map(s,d)")
ax.grid(True)
ax.legend()
fig.savefig(FIGURE_DIR / "trapetium_map.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot The Inverse Map Distance


In [ ]:
pad = 0.35
x_min = -0.5 * BOTTOM - pad
x_max = 0.5 * BOTTOM + pad
y_min = -0.5 * HEIGHT - pad
y_max = 0.5 * HEIGHT + pad

nx = 360
ny = 260
xg = np.linspace(x_min, x_max, nx)
yg = np.linspace(y_min, y_max, ny)
Xg, Yg = np.meshgrid(xg, yg)
D = np.empty_like(Xg)
S = np.empty_like(Xg)

for iy in range(ny):
    for ix in range(nx):
        s, d = trapetium_map_inverse(Xg[iy, ix], Yg[iy, ix], TOP, BOTTOM, HEIGHT)
        S[iy, ix] = s
        D[iy, ix] = d

fig, ax = plt.subplots(figsize=(6.5, 4.8))
levels = np.linspace(np.nanmin(D), np.nanmax(D), 30)
cf = ax.contourf(Xg, Yg, D, levels=levels, cmap="RdBu")
ax.contour(Xg, Yg, D, levels=[0.0], colors="black", linewidths=1.4)
ax.plot(P[:, 0], P[:, 1], color="black", lw=1.0)
fig.colorbar(cf, ax=ax, label="signed distance d")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("trapetium_map_inverse(x,y): d")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / "trapetium_inverse_d.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot And Check The Jacobian Maps


In [ ]:
DET_J = np.empty_like(Xg)
DET_INV_J = np.empty_like(Xg)

for iy in range(ny):
    for ix in range(nx):
        s = S[iy, ix]
        d = D[iy, ix]
        DET_J[iy, ix] = trapetium_jacobian_det(s, d, TOP, BOTTOM, HEIGHT)
        DET_INV_J[iy, ix] = trapetium_jacobian_inverse_det(s, d, TOP, BOTTOM, HEIGHT)

inside = D >= 0.0
min_abs_det_j = np.min(np.abs(DET_J[inside]))
min_abs_det_inv_j = np.min(np.abs(DET_INV_J[inside]))

print(f"min |det(J)| inside = {min_abs_det_j:.12e}")
print(f"min |det(J^-1)| inside = {min_abs_det_inv_j:.12e}")
print(f"expected det(J) = perimeter = {_perimeter(TOP, BOTTOM, HEIGHT):.12e}")
print(f"expected det(J^-1) = 1/perimeter = {1.0 / _perimeter(TOP, BOTTOM, HEIGHT):.12e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im0 = axes[0].contourf(Xg, Yg, np.where(inside, DET_J, np.nan), levels=12, cmap="viridis")
axes[0].plot(P[:, 0], P[:, 1], color="black", lw=1.0)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title(r"$\det J$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(Xg, Yg, np.where(inside, DET_INV_J, np.nan), levels=12, cmap="viridis")
axes[1].plot(P[:, 0], P[:, 1], color="black", lw=1.0)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_title(r"$\det J^{-1}$")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.colorbar(im1, ax=axes[1])

fig.tight_layout()
fig.savefig(FIGURE_DIR / "trapetium_jacobian_determinants.png", dpi=300, bbox_inches="tight")
plt.show()


## Quick Forward-Inverse Check


In [ ]:
rng = np.random.default_rng(1)
ss_all = []
dd_all = []
for d in np.linspace(0.02, 0.25, 20):
    for edge_id in range(4):
        s0, s1 = _edge_s_interval(edge_id, d, TOP, BOTTOM, HEIGHT)
        ss = rng.uniform(s0, s1, 25)
        ss_all.extend(ss)
        dd_all.extend([d] * len(ss))

ss = np.array(ss_all)
dd = np.array(dd_all)
xy = np.array([trapetium_map(s, d, TOP, BOTTOM, HEIGHT) for s, d in zip(ss, dd)])
inv = np.array([trapetium_map_inverse(x, y, TOP, BOTTOM, HEIGHT) for x, y in xy])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(dd, inv[:, 1], s=4, alpha=0.4)
lo = min(dd.min(), inv[:, 1].min())
hi = max(dd.max(), inv[:, 1].max())
ax.plot([lo, hi], [lo, hi], color="black", lw=1.0)
ax.set_xlabel("input d")
ax.set_ylabel("inverse d")
ax.set_title("Distance consistency on valid branches")
ax.grid(True)
fig.savefig(FIGURE_DIR / "trapetium_forward_inverse_check.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Create particles inside the trapezium.
particle_radius = 0.015
N = 1200
safety = 1.05
max_trials = 200000
particle_seed = 2
inverse_tolerance = 1.0e-8


def minimum_neighbour_distance(x):
    if len(x) < 2:
        return np.inf, np.full(len(x), np.inf)

    d_min = np.inf
    nearest = np.full(len(x), np.inf)
    for i in range(len(x)):
        d2 = np.sum((x - x[i])**2, axis=1)
        d2[i] = np.inf
        nearest[i] = np.sqrt(np.min(d2))
        d_min = min(d_min, nearest[i])
    return d_min, nearest


def generate_trapetium_particles_no_overlap(
    N,
    TOP,
    BOTTOM,
    HEIGHT,
    particle_radius,
    safety=1.05,
    max_trials=200000,
    seed=1,
    inverse_tolerance=1.0e-8,
):
    rng = np.random.default_rng(seed)

    min_dist = safety * 2.0 * particle_radius
    min_dist2 = min_dist * min_dist
    inverse_tol2 = inverse_tolerance * inverse_tolerance

    x_min = -0.5 * BOTTOM + particle_radius
    x_max = 0.5 * BOTTOM - particle_radius
    y_min = -0.5 * HEIGHT + particle_radius
    y_max = 0.5 * HEIGHT - particle_radius

    q_acc = np.empty((N, 2), dtype=np.float64)
    x_acc = np.empty((N, 2), dtype=np.float64)

    count = 0
    trials = 0

    while count < N and trials < max_trials:
        trials += 1

        x_new = rng.uniform(x_min, x_max)
        y_new = rng.uniform(y_min, y_max)

        s_new, d_new = trapetium_map_inverse(x_new, y_new, TOP, BOTTOM, HEIGHT)

        if d_new <= particle_radius:
            continue

        x_check, y_check = trapetium_map(s_new, d_new, TOP, BOTTOM, HEIGHT)
        if (x_check - x_new)**2 + (y_check - y_new)**2 > inverse_tol2:
            continue

        if count == 0:
            accept = True
        else:
            d2 = np.sum((x_acc[:count] - np.array([x_new, y_new]))**2, axis=1)
            accept = np.min(d2) > min_dist2

        if accept:
            q_acc[count, 0] = s_new
            q_acc[count, 1] = d_new
            x_acc[count, 0] = x_new
            x_acc[count, 1] = y_new
            count += 1

    q_acc = q_acc[:count]
    x_acc = x_acc[:count]
    d_min, nearest = minimum_neighbour_distance(x_acc)

    info = {
        "requested": N,
        "placed": count,
        "trials": trials,
        "particle_diameter": 2.0 * particle_radius,
        "minimum_allowed_distance": min_dist,
        "minimum_actual_distance": d_min,
    }
    return q_acc, x_acc, nearest, info


q_particles, x_particles, nearest, particle_info = generate_trapetium_particles_no_overlap(
    N=N,
    TOP=TOP,
    BOTTOM=BOTTOM,
    HEIGHT=HEIGHT,
    particle_radius=particle_radius,
    safety=safety,
    max_trials=max_trials,
    seed=particle_seed,
    inverse_tolerance=inverse_tolerance,
)

q0 = q_particles.astype(np.float32)
p0 = np.zeros_like(q0, dtype=np.float32)
x0 = x_particles.astype(np.float32)

print(particle_info)

s_boundary = np.linspace(0.0, 1.0, 1000)
boundary = np.array([trapetium_border(s, TOP, BOTTOM, HEIGHT) for s in s_boundary])

fig, (ax_phys, ax_q) = plt.subplots(1, 2, figsize=(10, 4))

ax_phys.plot(boundary[:, 0], boundary[:, 1], color="black", lw=1.0)
ax_phys.scatter(x_particles[:, 0], x_particles[:, 1], s=0.5)
ax_phys.set_aspect("equal", adjustable="box")
ax_phys.set_xlabel("x")
ax_phys.set_ylabel("y")
ax_phys.set_title("physical space")
ax_phys.grid(True)

ax_q.scatter(q_particles[:, 0], q_particles[:, 1], s=1.0, color="red")
ax_q.axhline(0.0, color="black", lw=1.0)
ax_q.set_xlim(0.0, 1.0)
ax_q.set_ylim(0.0, 1.05 * q_particles[:, 1].max())
ax_q.set_xlabel("s")
ax_q.set_ylabel("d")
ax_q.set_title("signed-distance coordinates")
ax_q.grid(True)

fig.savefig(FIGURE_DIR / "trapetium_initial_particles.png", dpi=300, bbox_inches="tight")
plt.show()


## Notes

The IQ signed distance used in the inverse map is unchanged except for the sign convention (`d > 0` inside). The side-branch forward map is valid only where that side is the nearest wall feature. For a positive distance `d`, the valid interval on each side is shorter than the original side; near corners the adjacent side becomes the nearest feature. The graph therefore uses `_edge_s_interval` to plot only the valid injective branch.

On each valid side branch, `J` is constant and non-singular. Because `s` is a perimeter fraction, `det(J)` equals the perimeter. If local arc length were used instead, `det(J)` would be 1.
